# 1. 读写文件
到目前为止，我们讨论了如何处理数据以及如何构建、训练和测试深度学习模型。然而，有时我们希望保存的训练的模型，以备将来在各种环境中使用。**当执行一个耗时较长的训练过程时，最佳的做法是定期保存中间结果，以防在服务器断电时，我们不会损失数天的计算结果。**

## 1.1 加载和保存张量
对于单个张量，我们可以直接调用load和save函数分别读写它们。这两个函数都要求我们提供一个名称，save要求将要保存的变量作为输入。

In [5]:
import torch
from torch import nn
from torch.nn import functional as F

x = torch.arange(4)
torch.save(x, 'x-file')

In [6]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

In [7]:
# 还可以存储一个张量列表，然后读回内存
y = torch.zeros(4)
torch.save([x, y], 'x-files')

In [8]:
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

我们还可以读取或写入从字符串映射到张量的字典。当我们要读取或写入模型中所有权重时，这就很方便。

In [9]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

## 加载和保存模型参数
保存单个权重向量确实有用，但是如果我们想保存整个模型并在以后加载它们，单独保存每个向量会变得很麻烦。毕竟，我们可能有数百个参数散布在各处。因此，深度学习框架提供了内置函数来保存和加载整个网络。需要注意的一个重要细节是，**这将保存模型的参数而不是整个模型。** 例如，如果有一个3层的多层感知机，我们需要单独指定架构。因为模型可以包含任意代码，所以模型本身难以序列化。因此，为了恢复模型，我们需要用代码生成架构，然后从磁盘加载参数。

In [10]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size = (2, 20))
Y = net(X)

接下来，我们将模型的参数存储在一个叫做mlp.params的文件中。

In [11]:
torch.save(net.state_dict(), 'mlp.params')

In [12]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)

由于两个实例具有相同的模型参数，在输入相同的X时，两个实例的计算结果应该相同。

In [13]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])